In [26]:
import numpy as np
import pandas as pd

In [2]:
property_sales = pd.read_csv('datasets/estimate-the-estimate/manhattan_property_sales.csv')

In [4]:
property_sales

,NEIGHBORHOOD,ADDRESS,ZIP_CODE,BUILDING_CLASS,SQUARE_FEET,SALE_PRICE
0,CHELSEA,150 WEST 15TH STREET,10011,A4,8997,14999999
1,CHELSEA,348 WEST 22 STREET,10011,A4,3042,13500000
2,CHELSEA,204 WEST 21ST STREET,10011,A4,3365,7650000
3,CHELSEA,481 WEST 22 STREET,10011,A4,3666,0
4,CHELSEA,344 W 22 STREET,10011,A4,4395,13100000
...,...,...,...,...,...,...
136,UPPER WEST SIDE,324 WEST 85 STREET,10024,A4,2976,3500000
137,UPPER WEST SIDE,324 WEST 85 STREET,10024,A4,2976,0
138,UPPER WEST SIDE,52 WEST 84TH STREET,10024,A5,3651,0
139,UPPER WEST SIDE,315 WEST 84 STREET,10024,A5,4078,6500000


In [30]:
avg_sales_per_zip = property_sales.groupby(['ZIP_CODE', 'BUILDING_CLASS'], as_index=False) \
    .apply(lambda x: (x['SALE_PRICE'] / x['SQUARE_FEET']).mean().round())

/tmp/ipykernel_51/3352840861.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x['SALE_PRICE'] / x['SQUARE_FEET']).mean().round())


In [31]:
df_new = property_sales.merge(
    avg_sales_per_zip,
    how='inner',
    on=['ZIP_CODE', 'BUILDING_CLASS']
)
df_new.rename(columns={None: 'avg_sales'}, inplace=True)

In [32]:
df_new

,NEIGHBORHOOD,ADDRESS,ZIP_CODE,BUILDING_CLASS,SQUARE_FEET,SALE_PRICE,avg_sales
0,CHELSEA,150 WEST 15TH STREET,10011,A4,8997,14999999,2075.0
1,CHELSEA,348 WEST 22 STREET,10011,A4,3042,13500000,2075.0
2,CHELSEA,204 WEST 21ST STREET,10011,A4,3365,7650000,2075.0
3,CHELSEA,481 WEST 22 STREET,10011,A4,3666,0,2075.0
4,CHELSEA,344 W 22 STREET,10011,A4,4395,13100000,2075.0
...,...,...,...,...,...,...,...
136,UPPER WEST SIDE,324 WEST 85 STREET,10024,A4,2976,3500000,974.0
137,UPPER WEST SIDE,324 WEST 85 STREET,10024,A4,2976,0,974.0
138,UPPER WEST SIDE,52 WEST 84TH STREET,10024,A5,3651,0,797.0
139,UPPER WEST SIDE,315 WEST 84 STREET,10024,A5,4078,6500000,797.0


In [34]:
df_new['market_value'] = np.where(
    df_new['SALE_PRICE'] == 0,
    df_new['SQUARE_FEET'] * df_new['avg_sales'],
    df_new['SALE_PRICE']
)

In [36]:
df_new.tail(20)

,NEIGHBORHOOD,ADDRESS,ZIP_CODE,BUILDING_CLASS,SQUARE_FEET,SALE_PRICE,avg_sales,market_value
121,UPPER EAST SIDE,131 EAST 95 STREET,10128,A9,3536,9500000,1891.0,9500000.0
122,UPPER EAST SIDE,532 EAST 87TH STREET,10128,A9,3168,0,1891.0,5990688.0
123,UPPER EAST SIDE,131 EAST 92 STREET,10128,A9,3946,8325000,1891.0,8325000.0
124,UPPER WEST SIDE,161 WEST 73 STREET,10023,A4,5467,5700000,1120.0,5700000.0
125,UPPER WEST SIDE,311 WEST 74 STREET,10023,A4,6380,7650000,1120.0,7650000.0
126,UPPER WEST SIDE,131 WEST 70 STREET,10023,A4,6956,4320000,1120.0,4320000.0
127,UPPER WEST SIDE,21 WEST 70TH STREET,10023,A4,6450,8180000,1120.0,8180000.0
128,UPPER WEST SIDE,226 WEST 71ST STREET,10023,A4,7820,11500000,1120.0,11500000.0
129,UPPER WEST SIDE,37 WEST 74 STREET,10023,A9,6576,9800000,1490.0,9800000.0
130,UPPER WEST SIDE,38 WEST 87TH STREET,10024,A4,6254,14600000,974.0,14600000.0


After imputing, how many total properties have an estimated market value greater than $15 million?

In [39]:
len(df_new.query('market_value > 15_000_000'))

19